# DINOv3 for SLAM — 1. Loop closure

**Question:** can DINOv3 tokens recognise a revisited place well enough to trigger loop closures?

**Data:** TUM RGB-D `fr3_long_office_household` — a hand-held camera circles two desks and returns to its start, so the end of the sequence revisits the beginning.

**Protocol**
* Every `STRIDE`-th frame is both a query and a database entry. A query may only match frames at least `EXCLUDE_S` seconds in its **past** (as in online SLAM, and so trivially adjacent frames do not count).
* Ground-truth loop: a candidate within `D_THR` metres and `ANG_THR` degrees of the query (from the motion-capture poses).
* Descriptors (all training-free on top of frozen DINOv3):
  * `cls`: the CLS token
  * `mean` / `gem`: average or GeM pooling of the patch tokens
  * `cls+mean`: both, concatenated
  * `vlad`: VLAD over patch tokens with a 32-word vocabulary built on a *different* sequence (`fr1_desk`), as in AnyLoc
* Baseline: an ORB bag of words (1000 words, TF-IDF), with the vocabulary also built on `fr1_desk`. This is a flat-vocabulary stand-in for DBoW2 as used in ORB-SLAM.

**Metrics**
* **Recall@1**: of the queries that have a true loop, the fraction whose top-1 retrieval is correct.
* **PR-AUC (AP)**: each query proposes its top-1 match with a score, and we sweep a threshold over that score. Queries without a true loop count as false positives when they fire, which is the case that matters for SLAM.
* **Recall @ 100% precision**: how many loops we could accept before the first false one. A single false loop closure can wreck a pose graph.
* Last, a re-ranking step that uses **geometric verification with DINOv3 patch matches + depth + PnP**, like the verification step in a real SLAM loop-closure module.

In [ ]:
%load_ext autoreload
%autoreload 2
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import trange, tqdm

import slam_utils as su

SEQ = "fr3_long_office_household"
VOCAB_SEQ = "fr1_desk"        # vocabularies (ORB BoW, VLAD) are trained here: no leakage
MODEL = "vits16"              # vits16 | vits16plus | vitb16 | vitl16
STRIDE = 5                    # 30 Hz -> 6 Hz keyframes
SCALE = 1.0                   # 640x480 -> 40x30 patch grid
EXCLUDE_S = 20.0
D_THR, ANG_THR = 0.5, 30.0
DATA = su.DATA_ROOT  # TUM sequences are downloaded here on first use

seq = su.TUMSequence(SEQ, DATA)
vocab_seq = su.TUMSequence(VOCAB_SEQ, DATA)
frames = np.arange(0, len(seq), STRIDE)
print(seq, "\n", vocab_seq, "\n", len(frames), "keyframes")

## Ground truth loops

In [ ]:
poses = seq.poses[frames]
ts = seq.timestamps[frames]
cand, pos = su.loop_ground_truth(poses, ts, D_THR, ANG_THR, EXCLUDE_S, causal=True)
has_loop = pos.any(1)
print(f"{has_loop.sum()} / {len(frames)} queries have a true loop in their past")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
P = poses[:, :3, 3]
sc = ax[0].scatter(P[:, 0], P[:, 1], c=ts - ts[0], s=6, cmap="viridis")
ax[0].scatter(P[has_loop, 0], P[has_loop, 1], facecolors="none", edgecolors="r", s=30, lw=0.6, label="query with a true loop")
ax[0].set_aspect("equal"); ax[0].set_title("Ground-truth trajectory (top view)"); ax[0].legend()
plt.colorbar(sc, ax=ax[0], label="time [s]")
ax[1].imshow(pos, cmap="gray_r", interpolation="nearest")
ax[1].set_title("True loops: query (row) vs past keyframe (col)"); ax[1].set_xlabel("database"); ax[1].set_ylabel("query")
plt.tight_layout(); plt.show()

## Extract DINOv3 features
We keep the CLS token and the full patch grid in float16, which is enough for re-ranking later.

In [ ]:
dino = su.DinoV3(MODEL)
print(dino.name, "on", dino.device, "| dim", dino.dim, "| registers", dino.n_reg)

def extract(seq, idx, scale=SCALE, layer=None, bs=8):
    cls, pat = [], []
    for i in trange(0, len(idx), bs, desc=f"DINOv3 {seq.name}"):
        c, p, stride = dino([seq.rgb(f) for f in idx[i:i + bs]], scale=scale, layer=layer)
        cls.append(c); pat.append(p.astype(np.float16))
    return np.concatenate(cls), np.concatenate(pat), stride

t0 = time.time()
CLS, PAT, STRIDE_PX = extract(seq, frames)
print(f"{CLS.shape=} {PAT.shape=} | {1000 * (time.time() - t0) / len(frames):.0f} ms/frame")

A quick look at what the patch tokens encode: the first three PCA components shown as RGB, with one PCA basis shared across frames so colours are comparable. Frames from the start and the end of the loop should show the same objects in the same colours.

In [ ]:
show = [0, len(frames) // 3, 2 * len(frames) // 3, len(frames) - 1]
_, basis = su.pca_rgb(PAT[show].reshape(-1, 1, PAT.shape[-1]).astype(np.float32))
fig, ax = plt.subplots(2, len(show), figsize=(4 * len(show), 6))
for k, i in enumerate(show):
    ax[0, k].imshow(seq.rgb(frames[i])); ax[0, k].set_title(f"t={ts[i] - ts[0]:.1f}s"); ax[0, k].axis("off")
    ax[1, k].imshow(su.pca_rgb(PAT[i].astype(np.float32), basis)[0]); ax[1, k].axis("off")
plt.tight_layout(); plt.show()

## Global descriptors

In [ ]:
desc = {}
pooled = [su.pool_descriptors(c, p.astype(np.float32)) for c, p in zip(CLS, PAT)]
for k in pooled[0]:
    desc[f"DINOv3 {k}"] = np.stack([d[k] for d in pooled])

# VLAD vocabulary from a different sequence
vfr = np.arange(0, len(vocab_seq), 5)
_, VPAT, _ = extract(vocab_seq, vfr)
X = su.l2n(VPAT.reshape(-1, VPAT.shape[-1]).astype(np.float32))
X = X[np.random.default_rng(0).choice(len(X), min(len(X), 100_000), replace=False)]
km = su.fit_kmeans(X, 32)
desc["DINOv3 vlad"] = np.stack([su.vlad(p.astype(np.float32), km.cluster_centers_) for p in tqdm(PAT, desc="VLAD")])
del VPAT, X

# ORB bag of words baseline (vocabulary from a different sequence, IDF from this database)
bow = su.OrbBoW(k=1000, n_features=1000).fit_vocab([vocab_seq.rgb(f) for f in tqdm(vfr, desc="ORB vocab")])
TF = np.stack([bow.tf(seq.rgb(f)) for f in tqdm(frames, desc="ORB BoW")])
bow.fit_idf(TF)
desc["ORB BoW"] = bow.encode(TF)

{k: v.shape for k, v in desc.items()}

## Results

In [ ]:
results = {name: su.evaluate_retrieval(D @ D.T, cand, pos) for name, D in desc.items()}
table = pd.DataFrame({n: {k: r[k] for k in ["R@1", "R@5", "R@10", "AP", "R@100P"]} for n, r in results.items()}).T
table.style.format("{:.3f}").background_gradient(cmap="Greens", axis=0)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name, r in results.items():
    p, rc = r["curve"]
    ls = "--" if name.startswith("ORB") else "-"
    ax[0].plot(rc, p, ls, label=f"{name} (AP={r['AP']:.2f})")
ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision"); ax[0].set_title("Top-1 loop proposals: precision-recall")
ax[0].legend(fontsize=8); ax[0].set_ylim(0, 1.02)
ks = np.arange(1, 26)
for name, D in desc.items():
    r = su.evaluate_retrieval(D @ D.T, cand, pos, ks=tuple(ks))
    ax[1].plot(ks, [r[f"R@{k}"] for k in ks], "--" if name.startswith("ORB") else "-", label=name)
ax[1].set_xlabel("N"); ax[1].set_ylabel("Recall@N"); ax[1].set_title("Recall@N"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Similarity structure
A good descriptor should look like the ground-truth loop matrix above: a bright band in the bottom-left corner where the end of the trajectory revisits the start.

In [ ]:
names = ["DINOv3 cls", "DINOv3 gem", "DINOv3 vlad", "ORB BoW"]
fig, ax = plt.subplots(1, len(names), figsize=(4.2 * len(names), 4))
for a, n in zip(ax, names):
    D = desc[n]; S = D @ D.T
    a.imshow(S, cmap="magma", vmin=np.percentile(S, 5), vmax=np.percentile(S, 99.5))
    a.set_title(n); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

### Score separability
Can a single threshold tell true loops from false ones? Compare the top-1 similarity of correct and incorrect proposals.

In [ ]:
fig, ax = plt.subplots(1, len(names), figsize=(4.2 * len(names), 3.2))
for a, n in zip(ax, names):
    r = results[n]
    ok = np.isfinite(r["top1_score"])
    a.hist(r["top1_score"][ok & r["top1_correct"]], 30, alpha=0.6, label="correct", density=True)
    a.hist(r["top1_score"][ok & ~r["top1_correct"]], 30, alpha=0.6, label="wrong", density=True)
    a.set_title(n); a.set_xlabel("top-1 cosine similarity")
ax[0].legend(); plt.tight_layout(); plt.show()

### Qualitative examples
Each pair shows a loop query, then the top-1 retrievals from the best DINOv3 descriptor and from ORB BoW, each titled with its true distance from the query.

In [ ]:
best = table.drop("ORB BoW")["AP"].idxmax()
print("best DINOv3 descriptor:", best)
q_ids = np.nonzero(has_loop)[0]
q_ids = q_ids[np.linspace(0, len(q_ids) - 1, 4).astype(int)]
fig, ax = plt.subplots(len(q_ids), 3, figsize=(12, 3.2 * len(q_ids)))
for row, q in enumerate(q_ids):
    ax[row, 0].imshow(seq.rgb(frames[q])); ax[row, 0].set_title(f"query t={ts[q] - ts[0]:.0f}s")
    for col, n in [(1, best), (2, "ORB BoW")]:
        j = results[n]["top1"][q]
        d = np.linalg.norm(P[q] - P[j])
        okc = pos[q, j]
        ax[row, col].imshow(seq.rgb(frames[j]))
        ax[row, col].set_title(f"{n}: t={ts[j] - ts[0]:.0f}s, {d:.2f} m", color="g" if okc else "r")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## Geometric verification with DINOv3 patch matches
A real SLAM system does not trust appearance alone. Here we take the top-`K` candidates from the best global descriptor, match patch tokens (mutual nearest neighbours), lift the candidate's patches to 3D with its depth, and run PnP-RANSAC against the query. The new score is the **number of PnP inliers**. This reuses the patch tokens we already have, so it needs no new forward passes.

In [ ]:
K_RR, REPROJ = 5, 8.0  # 8 px ~ half a patch: coarse patch matches are quantised to 16 px
h, w = PAT.shape[1:3]
centers = su.patch_centers(h, w, STRIDE_PX)
grid = su.l2n(PAT.reshape(len(PAT), h * w, -1).astype(np.float32))
S0 = np.where(cand, desc[best] @ desc[best].T, -np.inf)
S_rr = np.full_like(S0, -np.inf)
depth_cache = {}
for q in tqdm(np.nonzero(cand.any(1))[0], desc="re-rank"):
    for j in np.argsort(-S0[q])[:K_RR]:
        if not np.isfinite(S0[q, j]):
            continue
        ia, ib, _, _ = su.mutual_nn(grid[j], grid[q], min_sim=0.3)
        if j not in depth_cache:
            depth_cache[j] = seq.depth(frames[j])
        r = su.estimate_pose_pnp(centers[ia], centers[ib], depth_cache[j], seq.K, seq.dist, reproj_thr=REPROJ, iters=500)
        S_rr[q, j] = r["inliers"].sum() + 1e-3 * S0[q, j]  # tie-break by appearance
results[f"{best} + geom. verif."] = su.evaluate_retrieval(S_rr, cand, pos)
table = pd.DataFrame({n: {k: r[k] for k in ["R@1", "R@5", "AP", "R@100P"]} for n, r in results.items()}).T
table.style.format("{:.3f}").background_gradient(cmap="Greens", axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for name in [best, "ORB BoW", f"{best} + geom. verif."]:
    p, rc = results[name]["curve"]
    ax.plot(rc, p, label=f"{name} (AP={results[name]['AP']:.2f})")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.legend(); ax.set_ylim(0, 1.02)
ax.set_title("Effect of geometric verification"); plt.show()

## Sensitivity
**1. Ground-truth threshold.** Loop recall depends on what counts as the "same place". A descriptor that is only good at near-duplicates loses more as the radius grows.

In [ ]:
rows = []
for d_thr in [0.25, 0.5, 1.0, 1.5]:
    c_, p_ = su.loop_ground_truth(poses, ts, d_thr, ANG_THR, EXCLUDE_S)
    for n in dict.fromkeys([best, "DINOv3 cls", "ORB BoW"]):
        r = su.evaluate_retrieval(desc[n] @ desc[n].T, c_, p_)
        rows.append({"D_THR [m]": d_thr, "descriptor": n, "R@1": r["R@1"], "AP": r["AP"], "#loops": r["n_queries_with_loop"]})
pd.DataFrame(rows).pivot(index="D_THR [m]", columns="descriptor", values=["R@1", "AP"]).round(3)

**2. Which layer and which resolution?** Intermediate layers are often better for retrieval and matching than the final layer. This re-extracts features, so it is slow on CPU; set `RUN_ABLATION = True` to run it.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    rows = []
    for layer in [dino.n_layers // 2, 3 * dino.n_layers // 4, None]:
        for scale in [0.5, 1.0]:
            c, p, _ = extract(seq, frames, scale=scale, layer=layer)
            pooled = [su.pool_descriptors(ci, pi.astype(np.float32)) for ci, pi in zip(c, p)]
            for k in ["cls", "gem"]:
                D = np.stack([d[k] for d in pooled])
                r = su.evaluate_retrieval(D @ D.T, cand, pos)
                rows.append({"layer": layer or dino.n_layers, "scale": scale, "desc": k, "R@1": r["R@1"], "AP": r["AP"]})
    display(pd.DataFrame(rows).pivot_table(index=["layer", "scale"], columns="desc", values=["R@1", "AP"]).round(3))

## Takeaways (fill in after running)
* Do **pooled patch tokens beat the CLS token**? The CLS token is trained for image-level semantics. Pooled or VLAD-aggregated patches keep more layout and instance detail.
* How does DINOv3 compare to **ORB BoW**, especially on **R@100P**, the number that decides whether loops can be accepted without verification?
* How much does **geometric verification** with the same patch tokens buy? This tells you whether one backbone can cover both retrieval and verification.